# x7_custo-milho-s2 — Custos de Produção de Milho 2ª Safra / Safrinha CONAB (Série MT)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** custos de produção de milho 2ª safra, unidade a conferir por aba, localidades de MT, safra/ano*  
**Conceito e Mecanismo Econômico:** Custos de produção de milho safrinha (cultivado imediatamente após a soja). Sistema dominante no Centro-Oeste brasileiro.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__costs_conab__milho_2a_safra_serie_historica_2005-2025__MT_celulas.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Manter safras separadas; não atribuir custos locais a todos os municípios sem justificativa.
> - Candidatos para explorar custos/margens; ainda não permitem medir diretamente retorno líquido esperado.

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`arquivo_origem`**: Planilha CONAB.
- **`aba`**: Aba.
- **`conteudo`**: Dado da célula.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__costs_conab__milho_2a_safra_serie_historica_2005-2025__MT_celulas.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x7_custo-milho-s2`:
- **Atenção:** Manter safras separadas; não atribuir custos locais a todos os municípios sem justificativa.
- **Atenção:** Candidatos para explorar custos/margens; ainda não permitem medir diretamente retorno líquido esperado.


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
# REGRA DO TXT: 'Manter safras separadas; não atribuir custos locais a todos os municípios sem justificativa.'
print(f"Abas de milho safrinha (2ª safra): {df['aba'].nunique()}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print(df.head())

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
df['aba'].value_counts().plot(kind='bar')
plt.title("Células por Praça/Ano Milho 2ª Safra")
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
print(df['aba'].unique())

## Fase 6: Feature Engineering — Custos de Milho 2ª Safra (Safrinha CONAB)

### 1. Step 1: Custos da Segunda Safra (Outono/Inverno)
- A safrinha de milho (plantada imediatamente após a colheita precoce da soja) é o motor da segunda safra em MT.
- Custos: COE, COT e CT em R$/ha e R$/saca.

### 2. Step 2: Razão de Custos Operacionais Safrinha vs Soja
- $\frac{COE_{milho\_s2}}{COE_{soja}}$: mede a viabilidade financeira do sistema de duplo cultivo. Se o custo da safrinha sobe muito, o agricultor pode abrir mão do segundo plantio.
- Ressalva de restrição espacial aos municípios-polo da CONAB.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# Estruturação de Custos de Milho Safrinha
df_m2 = df.copy()
print("Série de Custos de Milho 2ª Safra (Safrinha) CONAB:")
print(df_m2.head())
print("\nRazão de custos safrinha/soja orienta a viabilidade econômica do sistema de sucessão de culturas.")